In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D1 — CEDEFOP Labour Skills Shortage Dataset
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

import json
import hashlib
import math
import openpyxl
import pandas as pd

from pathlib import Path
from google.colab import files

DOCUMENT_ID = "D1"
DOCUMENT_NAME = "CEDEFOP Labour Skills Shortage Dataset"

BRANCH_ID = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "Workbook-to-Markdown structural conversion with "
    "merged-cell hierarchy expansion"
)

EXPECTED_SOURCE_FORMAT = ".xlsx"

EXPECTED_SOURCE_SHA256 = "4a0b8117c9abdaa0daeb002455fdda840f6149bf1f68096d33fa4744b975e389"

EXPECTED_SHEETS = ["EU27", "IT", "NL", "PT"]
EXPECTED_RECORD_COUNT = 156

EXPECTED_RECORD_FIELDS = [
    "Geographic Area",
    "Main Occupation Group",
    "Occupation Group (2 digit)",
    "Labour Shortage Index",
    "LSI (Comp.)",
    "LSI1",
    "LSI2",
    "LSI3"
]

NUMERIC_FIELDS = [
    "Labour Shortage Index",
    "LSI1",
    "LSI2",
    "LSI3"
]

EXPECTED_SOURCE_HEADERS = [
    "Main Occupation Group",
    "Occupation Group (2 digit)",
    "Labour Shortage Indexx",
    "LSI (Comp.)",
    "LSI1",
    "LSI2",
    "LSI3"
]

OUTPUT_DIR = Path("outputs_D1_branch_B")
OUTPUT_DIR.mkdir(exist_ok=True)

print("D1 Branch B configured.")
print("Frozen expected record count:", EXPECTED_RECORD_COUNT)


In [ ]:
# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Upload exactly one source document for D1.")

FILE_PATH = Path(next(iter(uploaded)))

print(f"Loaded source file: {FILE_PATH}")


In [ ]:
# ------------------------------------------------------------
# 2. Source document upload
# ------------------------------------------------------------

if FILE_PATH.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_FORMAT} source file, "
        f"received {FILE_PATH.suffix}"
    )

def calculate_sha256(file_path, chunk_size=8192):
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            sha256.update(chunk)
    return sha256.hexdigest()

SOURCE_SHA256 = calculate_sha256(FILE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

print("Observed source SHA-256:", SOURCE_SHA256)
print("Matches frozen D1 source:", SOURCE_HASH_MATCH)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "The uploaded XLSX does not match the frozen D1 source identity. "
        "Do not continue until the source version is reconciled."
    )


In [ ]:
# ------------------------------------------------------------
# 3. Source identity verification
# ------------------------------------------------------------

workbook = openpyxl.load_workbook(
    FILE_PATH,
    data_only=True
)

observed_sheets = workbook.sheetnames

if observed_sheets != EXPECTED_SHEETS:
    raise ValueError(
        "Unexpected D1 worksheet structure.\n"
        f"Expected: {EXPECTED_SHEETS}\n"
        f"Observed: {observed_sheets}"
    )

source_structure = []

for ws in workbook.worksheets:
    observed_headers = [
        ws.cell(row=1, column=col).value
        for col in range(1, ws.max_column + 1)
    ]

    if observed_headers != EXPECTED_SOURCE_HEADERS:
        raise ValueError(
            f"Unexpected headers in worksheet {ws.title}.\n"
            f"Expected: {EXPECTED_SOURCE_HEADERS}\n"
            f"Observed: {observed_headers}"
        )

    source_structure.append({
        "sheet_name": ws.title,
        "max_row_including_header": int(ws.max_row),
        "data_rows": int(ws.max_row - 1),
        "max_column": int(ws.max_column),
        "merged_ranges": [str(rng) for rng in ws.merged_cells.ranges]
    })

source_structure_df = pd.DataFrame(source_structure)
source_structure_df


In [ ]:
# ------------------------------------------------------------
# 4. Source-structure verification
# ------------------------------------------------------------

def merged_anchor_value(ws, row, col):
    """Return the source value governing a cell, resolving Excel merges."""
    cell = ws.cell(row=row, column=col)

    if cell.value is not None:
        return cell.value, False

    coordinate = cell.coordinate

    for merged_range in ws.merged_cells.ranges:
        if coordinate in merged_range:
            anchor = ws.cell(
                row=merged_range.min_row,
                column=merged_range.min_col
            )
            return anchor.value, True

    return None, False


converted_rows = []
conversion_log = []

for ws in workbook.worksheets:
    sheet_rows = []
    reconstructed_hierarchy_cells = 0

    for row_idx in range(2, ws.max_row + 1):
        raw_values = [
            ws.cell(row=row_idx, column=col).value
            for col in range(1, ws.max_column + 1)
        ]

        # Do not carry wholly empty source rows into the representation.
        if all(value is None for value in raw_values):
            continue

        main_group, reconstructed = merged_anchor_value(
            ws, row_idx, 1
        )
        reconstructed_hierarchy_cells += int(reconstructed)

        row_record = {
            "Geographic Area": ws.title,
            "Main Occupation Group": main_group,
            "Occupation Group (2 digit)": ws.cell(row=row_idx, column=2).value,
            # Preserve the source header spelling in the converted artefact.
            "Labour Shortage Indexx": ws.cell(row=row_idx, column=3).value,
            "LSI (Comp.)": ws.cell(row=row_idx, column=4).value,
            "LSI1": ws.cell(row=row_idx, column=5).value,
            "LSI2": ws.cell(row=row_idx, column=6).value,
            "LSI3": ws.cell(row=row_idx, column=7).value,
            "_source_row": row_idx
        }

        sheet_rows.append(row_record)
        converted_rows.append(row_record)

    conversion_log.append({
        "sheet_name": ws.title,
        "source_data_rows": int(ws.max_row - 1),
        "converted_rows": int(len(sheet_rows)),
        "hierarchy_cells_made_explicit": int(reconstructed_hierarchy_cells),
        "worksheet_identity_made_explicit": True,
        "row_order_preserved": True,
        "header_labels_standardised": False,
        "text_normalisation_applied": False,
        "value_normalisation_applied": False,
        "semantic_inference_applied": False
    })

converted_df_audit = pd.DataFrame(converted_rows)
conversion_log_df = pd.DataFrame(conversion_log)

print("Converted records:", len(converted_df_audit))
conversion_log_df


In [ ]:
# ------------------------------------------------------------
# 5. Structural conversion
# ------------------------------------------------------------

integrity_issues = []

expected_per_sheet = {
    ws.title: int(
        sum(
            1
            for row_idx in range(2, ws.max_row + 1)
            if not all(
                ws.cell(row=row_idx, column=col).value is None
                for col in range(1, ws.max_column + 1)
            )
        )
    )
    for ws in workbook.worksheets
}

observed_per_sheet = (
    converted_df_audit
    .groupby("Geographic Area")
    .size()
    .to_dict()
)

if len(converted_df_audit) != EXPECTED_RECORD_COUNT:
    integrity_issues.append({
        "issue": "record_count_mismatch",
        "expected": EXPECTED_RECORD_COUNT,
        "observed": int(len(converted_df_audit))
    })

if observed_per_sheet != expected_per_sheet:
    integrity_issues.append({
        "issue": "worksheet_row_count_mismatch",
        "expected": expected_per_sheet,
        "observed": observed_per_sheet
    })

source_to_converted = {
    2: "Occupation Group (2 digit)",
    3: "Labour Shortage Indexx",
    4: "LSI (Comp.)",
    5: "LSI1",
    6: "LSI2",
    7: "LSI3"
}

for _, row in converted_df_audit.iterrows():
    ws = workbook[row["Geographic Area"]]
    source_row = int(row["_source_row"])

    for col_idx, converted_col in source_to_converted.items():
        source_value = ws.cell(row=source_row, column=col_idx).value
        converted_value = row[converted_col]

        source_missing = source_value is None
        converted_missing = pd.isna(converted_value)

        if source_missing and converted_missing:
            continue

        if source_value != converted_value:
            integrity_issues.append({
                "issue": "source_value_changed",
                "sheet": ws.title,
                "source_row": source_row,
                "field": converted_col,
                "source_value": source_value,
                "converted_value": converted_value
            })

for _, row in converted_df_audit.iterrows():
    ws = workbook[row["Geographic Area"]]
    source_row = int(row["_source_row"])
    expected_main_group, _ = merged_anchor_value(ws, source_row, 1)

    if row["Main Occupation Group"] != expected_main_group:
        integrity_issues.append({
            "issue": "hierarchy_reconstruction_mismatch",
            "sheet": ws.title,
            "source_row": source_row,
            "expected": expected_main_group,
            "observed": row["Main Occupation Group"]
        })

duplicate_keys = int(
    converted_df_audit.duplicated(
        subset=["Geographic Area", "Occupation Group (2 digit)"]
    ).sum()
)

missing_target_values = (
    converted_df_audit[
        [
            "Geographic Area",
            "Main Occupation Group",
            "Occupation Group (2 digit)",
            "Labour Shortage Indexx",
            "LSI (Comp.)",
            "LSI1",
            "LSI2",
            "LSI3"
        ]
    ]
    .isna()
    .sum()
    .to_dict()
)

conversion_integrity = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "expected_sheets": EXPECTED_SHEETS,
    "observed_sheets": observed_sheets,
    "worksheet_order_preserved": observed_sheets == EXPECTED_SHEETS,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "converted_record_count": int(len(converted_df_audit)),
    "record_count_matches_stage1_scope":
        int(len(converted_df_audit)) == EXPECTED_RECORD_COUNT,
    "expected_records_per_sheet": expected_per_sheet,
    "converted_records_per_sheet": {
        k: int(v) for k, v in observed_per_sheet.items()
    },
    "duplicate_observation_keys": duplicate_keys,
    "missing_values_by_field": {
        k: int(v) for k, v in missing_target_values.items()
    },
    "source_header_typo_preserved": (
        "Labour Shortage Indexx" in converted_df_audit.columns
    ),
    "semantic_normalisation_applied": False,
    "content_value_changes_detected": int(
        sum(issue["issue"] == "source_value_changed"
            for issue in integrity_issues)
    ),
    "hierarchy_reconstruction_issues": int(
        sum(issue["issue"] == "hierarchy_reconstruction_mismatch"
            for issue in integrity_issues)
    ),
    "integrity_issue_count": int(len(integrity_issues)),
    "integrity_issues": integrity_issues,
    "conversion_integrity_passed": len(integrity_issues) == 0
}

print(json.dumps(conversion_integrity, indent=2, ensure_ascii=False))

if not conversion_integrity["conversion_integrity_passed"]:
    raise ValueError(
        "Branch B conversion-integrity checks failed. "
        "Inspect conversion_integrity before continuing."
    )


In [ ]:
# ------------------------------------------------------------
# 6. Conversion-integrity diagnostics
# ------------------------------------------------------------

REPRESENTATION_COLUMNS = [
    "Main Occupation Group",
    "Occupation Group (2 digit)",
    "Labour Shortage Indexx",
    "LSI (Comp.)",
    "LSI1",
    "LSI2",
    "LSI3"
]

def markdown_cell(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""

    if isinstance(value, float):
        text = repr(value)
    else:
        text = str(value)

    return (
        text
        .replace("\\", "\\\\")
        .replace("|", "\\|")
        .replace("\r\n", "<br>")
        .replace("\n", "<br>")
        .replace("\r", "<br>")
    )

def dataframe_to_preserving_markdown(df, columns):
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"

    rows = []
    for _, record in df.iterrows():
        rows.append(
            "| "
            + " | ".join(markdown_cell(record[col]) for col in columns)
            + " |"
        )

    return "\n".join([header, separator] + rows)

markdown_sections = []

for sheet in EXPECTED_SHEETS:
    sheet_df = converted_df_audit[
        converted_df_audit["Geographic Area"] == sheet
    ].copy()

    section = (
        f"## Geographic Area: {sheet}\n\n"
        + dataframe_to_preserving_markdown(
            sheet_df,
            REPRESENTATION_COLUMNS
        )
    )

    markdown_sections.append(section)

BRANCH_B_REPRESENTATION = (
    "# CEDEFOP Labour Skills Shortage Index — Structural Representation\n\n"
    "The four original workbook worksheets are represented below as "
    "separate Geographic Area sections. Vertically merged values in "
    "`Main Occupation Group` are repeated only to make the source hierarchy "
    "explicit in a text-tabular representation. Source headers and source "
    "values are otherwise preserved.\n\n"
    + "\n\n".join(markdown_sections)
)

print(BRANCH_B_REPRESENTATION[:3000])
print("\n[Preview truncated]")


In [ ]:
# ------------------------------------------------------------
# 7. Branch B representation
# ------------------------------------------------------------

REPRESENTATION_PATH = (
    OUTPUT_DIR / "D1_branch_B_structured_representation.md"
)

CONVERTED_DATA_PATH = (
    OUTPUT_DIR / "D1_branch_B_converted_data_audit.csv"
)

CONVERSION_LOG_PATH = (
    OUTPUT_DIR / "D1_branch_B_conversion_log.csv"
)

CONVERSION_INTEGRITY_PATH = (
    OUTPUT_DIR / "D1_branch_B_conversion_integrity.json"
)

SOURCE_STRUCTURE_PATH = (
    OUTPUT_DIR / "D1_branch_B_source_structure.csv"
)

REPRESENTATION_PATH.write_text(
    BRANCH_B_REPRESENTATION,
    encoding="utf-8"
)

converted_df_audit.to_csv(
    CONVERTED_DATA_PATH,
    index=False
)

conversion_log_df.to_csv(
    CONVERSION_LOG_PATH,
    index=False
)

source_structure_df.to_csv(
    SOURCE_STRUCTURE_PATH,
    index=False
)

with open(
    CONVERSION_INTEGRITY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        conversion_integrity,
        f,
        indent=2,
        ensure_ascii=False
    )

REPRESENTATION_SHA256 = calculate_sha256(REPRESENTATION_PATH)

print("Representation saved:", REPRESENTATION_PATH)
print("Representation SHA-256:", REPRESENTATION_SHA256)


In [ ]:
# ------------------------------------------------------------
# 8. Preserve Branch B representation and conversion evidence
# ------------------------------------------------------------

branch_representation = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "representation_type": "Structurally converted Markdown",
    "source_file": FILE_PATH.name,
    "source_format": FILE_PATH.suffix.lower(),
    "source_sha256": SOURCE_SHA256,
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "branch_name": BRANCH_NAME,
    "conversion_method": CONVERSION_METHOD,
    "llm_input_representation":
        "Structural Markdown",
    "structural_conversion_applied": True,
    "normalisation_applied": False,
    "ocr_applied": False,
    "derived_representation_used_as_model_input": True,
    "structural_operations": [
        "Preserve worksheet order",
        "Represent each worksheet as a Geographic Area section",
        "Preserve source row order",
        "Expand vertically merged Main Occupation Group labels",
        "Represent source table structure in Markdown"
    ],
    "operations_explicitly_not_applied": [
        "Source-header correction or standardisation",
        "Unit conversion",
        "Value calculation or semantic repair",
        "Terminology harmonisation",
        "OCR"
    ],
    "conversion_integrity_passed":
        conversion_integrity["conversion_integrity_passed"],
    "model_input_description":
        "The derived Markdown representation is submitted to the LLM instead "
        "of the original XLSX workbook."
}

REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D1_branch_B_representation.json"
)

with open(
    REPRESENTATION_METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        branch_representation,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(branch_representation, indent=2, ensure_ascii=False))


In [ ]:
# ------------------------------------------------------------
# 9. Fixed extraction schema
# ------------------------------------------------------------

EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "records": [
        {
            "Geographic Area": None,
            "Main Occupation Group": None,
            "Occupation Group (2 digit)": None,
            "Labour Shortage Index": None,
            "LSI (Comp.)": None,
            "LSI1": None,
            "LSI2": None,
            "LSI3": None
        }
    ]
}

print(json.dumps(EXTRACTION_SCHEMA, indent=2, ensure_ascii=False))


In [ ]:
# ------------------------------------------------------------
# 10. Fixed extraction task
# ------------------------------------------------------------

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract the labour shortage information from the attached structurally
converted representation of the CEDEFOP Labour Skills Shortage Index workbook.

Process every Geographic Area section in the representation.

Return one record for each occupation-group observation.

For each record, extract:
- Geographic Area
- Main Occupation Group
- Occupation Group (2 digit)
- Labour Shortage Index
- LSI (Comp.)
- LSI1
- LSI2
- LSI3

Extraction rules:
- Extract only information explicitly supported by the provided representation.
- Use the Geographic Area section heading as the Geographic Area.
- Preserve the association between each geographic area, main occupation
  group, two-digit occupation group and corresponding LSI values.
- Preserve LSI (Comp.) exactly as represented.
- Return LSI1, LSI2 and LSI3 as numerical values.
- Return Labour Shortage Index as a numerical value.
- Use null only when a requested value is not available.
- Do not infer, calculate, reconstruct or invent missing values.
- Do not omit repeated Main Occupation Group values from individual records.
- Return only valid JSON.
- Do not include explanations before or after the JSON.
- Keep the exact field names defined in the schema.
"""

print(EXTRACTION_TASK)


In [ ]:
# ------------------------------------------------------------
# 11. Extraction prompt
# ------------------------------------------------------------

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(EXTRACTION_SCHEMA, indent=2, ensure_ascii=False)}

The structurally converted Markdown representation is attached
as the extraction source.

Return only the JSON object.
""".strip()

PROMPT_PATH = OUTPUT_DIR / "D1_branch_B_prompt.txt"
PROMPT_PATH.write_text(FULL_PROMPT, encoding="utf-8")

print(FULL_PROMPT)
print()
print("Prompt saved to:", PROMPT_PATH)


In [ ]:
# ------------------------------------------------------------
# 12. Experimental metadata
# ------------------------------------------------------------

experiment_metadata = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH_ID,
    "branch_name": BRANCH_NAME,
    "source_file": FILE_PATH.name,
    "source_format": FILE_PATH.suffix.lower(),
    "source_sha256": SOURCE_SHA256,
    "source_structure": {
        "expected_sheets": EXPECTED_SHEETS,
        "observed_sheets": observed_sheets,
        "sheet_structure_verified": observed_sheets == EXPECTED_SHEETS
    },
    "content_validation_performed":
        False,
    "representation_file":
        REPRESENTATION_PATH.name,
    "representation_sha256":
        REPRESENTATION_SHA256,
    "conversion_method":
        CONVERSION_METHOD,

    "llm_input_representation":
        "Structural Markdown",
    "structural_conversion_applied": True,
    "normalisation_applied": False,
    "ocr_applied": False,
    "conversion_integrity_passed":
        conversion_integrity["conversion_integrity_passed"],
    "expected_extraction_scope": {
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "expected_fields": EXPECTED_RECORD_FIELDS
    },
    "prompt_file": PROMPT_PATH.name,
    "execution_environment":
        "Independent ChatGPT conversation",
    "expected_output_format": "JSON",
    "notes": (
        "Only document representation changes relative to Branch A. "
        "The fixed extraction task and schema remain unchanged. "
        "No Stage 1 reference values are supplied to the model. "
        "Content-level validation is performed separately in "
        "Validation B — D1."
    )
}

METADATA_PATH = OUTPUT_DIR / "D1_branch_B_experiment_metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        experiment_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(experiment_metadata, indent=2, ensure_ascii=False))

for path in [
    REPRESENTATION_PATH,
    PROMPT_PATH,
    METADATA_PATH,
    REPRESENTATION_METADATA_PATH,
    CONVERSION_LOG_PATH,
    CONVERSION_INTEGRITY_PATH,
    SOURCE_STRUCTURE_PATH
]:
    files.download(path)

# Stage 3 — Controlled LLM extraction for Branch B

Run the extraction in the independent conversation:
D1_branch_B_structured_representation.md

Upload of the **complete untouched response** to continue.

The notebook must preserve the raw response before attempting JSON parsing. Invalid JSON is retained as an experimental outcome rather than manually repaired.

In [ ]:
# ------------------------------------------------------------
# 13. Raw response upload
# ------------------------------------------------------------

uploaded_output = files.upload()

if len(uploaded_output) != 1:
    raise ValueError(
        "Upload exactly one file containing the complete "
        "raw D1 Branch B LLM response."
    )

RAW_OUTPUT_PATH = Path(next(iter(uploaded_output)))

print("Uploaded model response:", RAW_OUTPUT_PATH)


In [ ]:
# ------------------------------------------------------------
# 14. Raw response before parsing
# ------------------------------------------------------------

RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D1_branch_B_raw_response.txt"
)

raw_response_text = RAW_OUTPUT_PATH.read_text(
    encoding="utf-8"
)

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = calculate_sha256(RAW_RESPONSE_PATH)

print(
    "Raw LLM response preserved exactly as supplied "
    "before parsing or validation."
)
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


In [ ]:
# ------------------------------------------------------------
# 15. Parsing of raw response without modifying its contents
# ------------------------------------------------------------

json_valid = False
json_error = None
raw_extraction = None

try:
    raw_extraction = json.loads(raw_response_text)
    json_valid = True
except json.JSONDecodeError as error:
    json_error = str(error)

print("Valid JSON:", json_valid)

if json_error:
    print("JSON parsing error:")
    print(json_error)


In [ ]:
# ------------------------------------------------------------
# 16. Check top-level output structure
# ------------------------------------------------------------

top_level_checks = {
    "output_is_json_object": False,
    "document_id_present": False,
    "document_id_correct": False,
    "branch_present": False,
    "branch_correct": False,
    "records_present": False,
    "records_is_list": False
}

if json_valid and isinstance(raw_extraction, dict):
    top_level_checks["output_is_json_object"] = True
    top_level_checks["document_id_present"] = (
        "document_id" in raw_extraction
    )
    top_level_checks["document_id_correct"] = (
        raw_extraction.get("document_id") == DOCUMENT_ID
    )
    top_level_checks["branch_present"] = (
        "branch" in raw_extraction
    )
    top_level_checks["branch_correct"] = (
        raw_extraction.get("branch") == BRANCH_ID
    )
    top_level_checks["records_present"] = (
        "records" in raw_extraction
    )
    top_level_checks["records_is_list"] = isinstance(
        raw_extraction.get("records"),
        list
    )

print(json.dumps(top_level_checks, indent=2))


In [ ]:
# ------------------------------------------------------------
# 17. Check extraction record count
# ------------------------------------------------------------

observed_record_count = None
record_count_matches = False

if (
    json_valid
    and isinstance(raw_extraction, dict)
    and isinstance(raw_extraction.get("records"), list)
):
    observed_record_count = len(raw_extraction["records"])
    record_count_matches = (
        observed_record_count == EXPECTED_RECORD_COUNT
    )

record_count_check = {
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "record_count_matches": record_count_matches
}

print(json.dumps(record_count_check, indent=2))


In [ ]:
# ------------------------------------------------------------
# 18. Check record-level field structure
# ------------------------------------------------------------

record_structure_issues = []

if (
    json_valid
    and isinstance(raw_extraction, dict)
    and isinstance(raw_extraction.get("records"), list)
):
    expected_field_set = set(EXPECTED_RECORD_FIELDS)

    for index, record in enumerate(raw_extraction["records"]):
        if not isinstance(record, dict):
            record_structure_issues.append({
                "record_index": index,
                "issue": "Record is not a JSON object"
            })
            continue

        actual_fields = set(record.keys())

        missing_fields = sorted(
            expected_field_set - actual_fields
        )
        additional_fields = sorted(
            actual_fields - expected_field_set
        )

        if missing_fields or additional_fields:
            record_structure_issues.append({
                "record_index": index,
                "missing_fields": missing_fields,
                "additional_fields": additional_fields
            })

print(
    "Records with structural issues:",
    len(record_structure_issues)
)


In [ ]:
# ------------------------------------------------------------
# 19. Check expected numeric field types
# ------------------------------------------------------------

field_type_issues = []

if (
    json_valid
    and isinstance(raw_extraction, dict)
    and isinstance(raw_extraction.get("records"), list)
):
    for index, record in enumerate(raw_extraction["records"]):
        if not isinstance(record, dict):
            continue

        for field in NUMERIC_FIELDS:
            if field not in record:
                continue

            value = record[field]

            if value is None:
                continue

            # bool is technically an int subclass in Python but is not an
            # acceptable numerical extraction value here.
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                field_type_issues.append({
                    "record_index": index,
                    "field": field,
                    "observed_type": type(value).__name__,
                    "observed_value": value
                })

print("Numeric field type issues:", len(field_type_issues))


In [ ]:
# ------------------------------------------------------------
# 20. Compile and preserve structural diagnostics
# ------------------------------------------------------------

record_schema_valid = (
    len(record_structure_issues) == 0
)

field_types_valid = (
    len(field_type_issues) == 0
)

structurally_evaluable = all([
    json_valid,
    top_level_checks["output_is_json_object"],
    top_level_checks["document_id_present"],
    top_level_checks["document_id_correct"],
    top_level_checks["branch_present"],
    top_level_checks["branch_correct"],
    top_level_checks["records_present"],
    top_level_checks["records_is_list"],
    record_schema_valid,
    field_types_valid
])

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,
    "branch":
        BRANCH_ID,
    "json_valid":
        json_valid,
    "json_error":
        json_error,
    "top_level_checks":
        top_level_checks,
    "record_count_check":
        record_count_check,
    "records_with_structure_issues":
        len(record_structure_issues),
    "record_structure_issues":
        record_structure_issues,
    "numeric_field_type_issues":
        len(field_type_issues),
    "field_type_issues":
        field_type_issues,
    "record_schema_valid":
        bool(record_schema_valid),
    "field_types_valid":
        bool(field_types_valid),
    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR / "D1_branch_B_technical_diagnostics.json"
)

with open(
    TECHNICAL_DIAGNOSTICS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        TECHNICAL_DIAGNOSTICS,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(TECHNICAL_DIAGNOSTICS, indent=2, ensure_ascii=False))


In [ ]:
# ------------------------------------------------------------
# 21. Preserve parsed extraction only when JSON is valid
# ------------------------------------------------------------

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D1_branch_B_parsed_extraction.json"
)

if json_valid:
    with open(
        PARSED_EXTRACTION_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            raw_extraction,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("Parsed extraction saved to:", PARSED_EXTRACTION_PATH)
else:
    print(
        "Parsed extraction was not created because "
        "the preserved raw response is not valid JSON."
    )


In [ ]:
# ------------------------------------------------------------
# 22. Create Branch B experiment summary
# ------------------------------------------------------------

experiment_summary = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "source_file": FILE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": (
        SOURCE_HASH_MATCH
        and observed_sheets == EXPECTED_SHEETS
    ),
    "input_representation":
        "Structurally converted Markdown",
    "representation_file":
        REPRESENTATION_PATH.name,
    "representation_sha256":
        REPRESENTATION_SHA256,
    "structural_conversion_applied": True,
    "normalisation_applied": False,
    "ocr_applied": False,
    "conversion_integrity_passed":
        conversion_integrity["conversion_integrity_passed"],
    "json_valid": json_valid,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "record_count_matches": record_count_matches,
    "structurally_evaluable":
        bool(structurally_evaluable),
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,
    "records_with_structure_issues":
        len(record_structure_issues),
    "field_type_issues":
        len(field_type_issues),
    "raw_response_preserved": True,
    "raw_response_sha256": RAW_RESPONSE_SHA256,
    "parsed_extraction_created": json_valid,
    "content_validation_performed":
        False,
}

SUMMARY_PATH = (
    OUTPUT_DIR / "D1_branch_B_experiment_summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        experiment_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(experiment_summary, indent=2, ensure_ascii=False))


In [ ]:
# ------------------------------------------------------------
# 23. Final artefact inventory
# ------------------------------------------------------------

outputs_created = [
    SOURCE_STRUCTURE_PATH,
    CONVERTED_DATA_PATH,
    CONVERSION_LOG_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    METADATA_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    SUMMARY_PATH
]

if json_valid:
    outputs_created.append(PARSED_EXTRACTION_PATH)

print("D1 Branch B outputs created:")
for output in outputs_created:
    print("-", output)


In [ ]:
# ------------------------------------------------------------
# 24. Download final Branch B artefacts
# ------------------------------------------------------------

for output in outputs_created:
    files.download(output)

print(
    "\nFor Stage 4 Branch B validation, use:\n"
    "- the fixed Stage 1 D1_reference_values.csv;\n"
    "- D1_branch_B_parsed_extraction.json (only if JSON-valid);\n"
    "- D1_branch_B_technical_diagnostics.json.\n\n"
    "Reuse the frozen D1 validation rules established in Branch A: "
    "alignment identity = Geographic Area + Occupation Group (2 digit); "
    "primary correctness fields and deterministic comparison rules remain unchanged."
)
